# Week 3 — Serving & Inference

### Turn an inert model into a live service — and understand every layer of latency

> **MLOps & Deployment: A From-Scratch Engineering Codex**
> Part of the applied-systems track. Like the rest of this curriculum, every
> abstraction here is *built before it is used*: we re-implement the core of each
> MLOps tool in plain Python/NumPy first, then map what we built onto the
> industry-standard equivalent so the magic is gone by the time you reach it.

---

## Learning objectives

By the end of this notebook you will be able to:

1. Implement an HTTP inference endpoint from raw sockets, then with a framework.
2. Distinguish online, batch, and streaming inference and pick the right one.
3. Build a dynamic batching scheduler and measure its throughput gain.
4. Reason quantitatively about the latency vs. throughput trade-off.
5. Define and compute the SLO metrics (p50/p95/p99 latency, QPS) that govern serving.

## Prerequisites

- Week 2 (a packaged model with a predict function).
- Basic networking concepts (TCP, HTTP request/response).

## How to read this notebook

Each section has three layers:

- **🧠 Theory** — the systems problem and *why* it exists.
- **🔧 From scratch** — a minimal, dependency-light implementation you fully own.
- **🏭 In practice** — how the real tool (MLflow, Docker, K8s, etc.) does the same thing, and what it adds.

Run cells top-to-bottom. Everything is self-contained and works on a CPU-only laptop.

In [1]:
# --- Standard setup ---------------------------------------------------------
from __future__ import annotations
import os, sys, json, time, math, hashlib, random, pathlib, tempfile, shutil
from dataclasses import dataclass, field, asdict
from typing import Any, Callable

import numpy as np

np.random.seed(0)
random.seed(0)

# A scratch workspace that we clean up between runs of this notebook.
WORK = pathlib.Path(tempfile.mkdtemp(prefix="mlops_week_"))
print("Working directory for this notebook:", WORK)

# --- Content-addressable hashing (introduced & explained in Week 0) ---------
# Re-defined here so each notebook is fully self-contained and runnable alone.
def hash_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()[:12]

def hash_obj(obj) -> str:
    canonical = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str)
    return hash_bytes(canonical.encode())

def hash_array(a: np.ndarray) -> str:
    return hash_bytes(a.tobytes() + str(a.shape).encode() + str(a.dtype).encode())

def capture_environment() -> dict:
    import platform
    return {"python": sys.version.split()[0], "platform": platform.platform(),
            "numpy": np.__version__}

Working directory for this notebook: /tmp/mlops_week_06j7cfda


## 1. 🧠 What 'serving' means

A trained model is a pure function `predict(X) -> y`. **Serving** is the
engineering of exposing that function to *other systems*, *over a network*, *under
a latency and availability contract*. Three regimes cover almost everything:

| Regime | Pattern | Latency need | Example |
|--------|---------|--------------|---------|
| **Online** | one request → one prediction, synchronously | milliseconds | fraud check at checkout |
| **Batch** | a big file → many predictions, offline | minutes/hours | nightly churn scoring |
| **Streaming** | unbounded event stream → predictions | sub-second, continuous | live anomaly detection |

This week focuses on **online serving**, because it's the hardest and exposes
every systems trade-off. We'll build it from the socket up so nothing is magic.

## 2. 🔧 The absolute bottom: HTTP from a raw socket

Before any framework, let's see what an HTTP server *is*: a process that accepts a
TCP connection, reads bytes that happen to follow the HTTP grammar, and writes
bytes back. We'll handle one request in-process to keep the notebook synchronous
and inspectable.

In [2]:
# A trivial model to serve (recreating Week 2's interface inline)
W = np.array([2.0, -1.0]); B = 0.5
def model_predict(X):
    X = np.atleast_2d(np.asarray(X, dtype=float))
    return (X @ W + B).tolist()

def parse_http_request(raw: bytes) -> dict:
    """Parse the minimal slice of HTTP we need: method, path, body."""
    head, _, body = raw.partition(b"\r\n\r\n")
    lines = head.split(b"\r\n")
    method, path, _ = lines[0].decode().split(" ")
    headers = {}
    for line in lines[1:]:
        if b":" in line:
            k, v = line.decode().split(":", 1)
            headers[k.strip().lower()] = v.strip()
    return {"method": method, "path": path, "headers": headers, "body": body.decode()}

def build_http_response(status: str, obj: dict) -> bytes:
    body = json.dumps(obj).encode()
    return (
        f"HTTP/1.1 {status}\r\n"
        f"Content-Type: application/json\r\n"
        f"Content-Length: {len(body)}\r\n"
        f"\r\n"
    ).encode() + body

# Simulate the bytes a client would send (POST /predict with a JSON body)
sample_body = json.dumps({"instances": [[0.5, 0.5], [1.0, 0.0]]})
raw_request = (
    f"POST /predict HTTP/1.1\r\n"
    f"Host: localhost\r\n"
    f"Content-Type: application/json\r\n"
    f"Content-Length: {len(sample_body)}\r\n"
    f"\r\n{sample_body}"
).encode()

req = parse_http_request(raw_request)
print("Parsed request:", {k: req[k] for k in ("method", "path")})
print("Body:", req["body"])

Parsed request: {'method': 'POST', 'path': '/predict'}
Body: {"instances": [[0.5, 0.5], [1.0, 0.0]]}


## 3. 🔧 A minimal serving application (router + handlers)

Now the application layer: a router that maps `(method, path)` to a handler, plus
the handlers that do input validation and call the model. This is what *every*
web framework gives you — we're just building it so the framework holds no
mysteries.

In [3]:
class InferenceApp:
    def __init__(self, predict_fn, n_features: int):
        self.predict_fn = predict_fn
        self.n_features = n_features
        self.routes = {
            ("GET", "/health"): self.health,
            ("POST", "/predict"): self.predict,
            ("GET", "/metrics"): self.metrics,
        }
        self.request_count = 0
        self.error_count = 0

    def handle(self, raw: bytes) -> bytes:
        req = parse_http_request(raw)
        handler = self.routes.get((req["method"], req["path"]))
        if handler is None:
            self.error_count += 1
            return build_http_response("404 Not Found", {"error": "no such route"})
        try:
            status, obj = handler(req)
            return build_http_response(status, obj)
        except Exception as e:               # never leak a stack trace to a client
            self.error_count += 1
            return build_http_response("400 Bad Request", {"error": str(e)})

    def health(self, req):
        return "200 OK", {"status": "healthy"}

    def metrics(self, req):
        return "200 OK", {"requests": self.request_count, "errors": self.error_count}

    def predict(self, req):
        self.request_count += 1
        payload = json.loads(req["body"])
        X = payload.get("instances")
        if X is None:
            raise ValueError("missing 'instances'")
        X = np.atleast_2d(np.asarray(X, dtype=float))
        if X.shape[1] != self.n_features:   # the Week 2 signature guard, at the edge
            raise ValueError(f"expected {self.n_features} features, got {X.shape[1]}")
        return "200 OK", {"predictions": self.predict_fn(X)}


app = InferenceApp(model_predict, n_features=2)

# Drive it with the raw request from above + a couple of edge cases
print("POST /predict ->", app.handle(raw_request).split(b"\r\n\r\n")[1].decode())
print("GET  /health  ->", app.handle(b"GET /health HTTP/1.1\r\n\r\n").split(b"\r\n\r\n")[1].decode())
print("GET  /metrics ->", app.handle(b"GET /metrics HTTP/1.1\r\n\r\n").split(b"\r\n\r\n")[1].decode())
bad = b'POST /predict HTTP/1.1\r\n\r\n{"instances": [[1,2,3]]}'
print("bad shape     ->", app.handle(bad).split(b"\r\n\r\n")[1].decode())

POST /predict -> {"predictions": [1.0, 2.5]}
GET  /health  -> {"status": "healthy"}
GET  /metrics -> {"requests": 1, "errors": 0}
bad shape     -> {"error": "expected 2 features, got 3"}


You now have a working inference service: routing, JSON parsing, input validation,
error handling, a health check, and a metrics endpoint. The health and metrics
endpoints aren't decoration — they're the contract the orchestrator (Week 2's
`HEALTHCHECK`, and Kubernetes later) uses to decide whether to send you traffic.

## 4. 🧠 The throughput problem, and the batching insight

Here's the systems crux of ML serving. A model's `predict` has two cost components:

$$ T(\text{batch of } b) \approx \underbrace{c_0}_{\text{fixed overhead}} + \underbrace{b \cdot c_1}_{\text{per-item}} $$

The fixed cost `c₀` (kernel launch, framework dispatch, memory setup) is paid
*per call*, not per item. So processing 32 requests **one at a time** pays `c₀`
32 times; processing them **as one batch of 32** pays `c₀` once. On a GPU `c₀`
dominates, so batching can yield near-linear throughput gains.

The tension: to batch, you must **wait** to collect requests — adding latency. The
art of online serving is **dynamic batching**: collect requests for a tiny window
(or until a max batch size), then run them together. We'll build it and measure
the trade-off.

In [4]:
# Model a predict cost with real fixed-overhead structure
FIXED_OVERHEAD = 0.010      # 10 ms paid per call regardless of batch size
PER_ITEM_COST  = 0.0005     # 0.5 ms per item

def timed_predict(batch: np.ndarray) -> np.ndarray:
    b = len(batch)
    time.sleep(FIXED_OVERHEAD + b * PER_ITEM_COST)   # simulate compute
    return batch @ W + B

# Naive: one request at a time
def serve_naive(requests):
    t0 = time.perf_counter()
    outs = [timed_predict(np.atleast_2d(r)) for r in requests]
    return outs, time.perf_counter() - t0

# Batched: all at once
def serve_batched(requests):
    t0 = time.perf_counter()
    batch = np.array(requests)
    outs = timed_predict(batch)
    return outs, time.perf_counter() - t0

reqs = [rng_ := np.random.rand(2) for _ in range(32)]
_, t_naive = serve_naive(reqs)
_, t_batch = serve_batched(reqs)
print(f"32 requests, one-at-a-time : {t_naive*1000:7.1f} ms  ({32/t_naive:6.1f} req/s)")
print(f"32 requests, single batch  : {t_batch*1000:7.1f} ms  ({32/t_batch:6.1f} req/s)")
print(f"Throughput speed-up        : {t_naive/t_batch:5.1f}x")

32 requests, one-at-a-time :   340.5 ms  (  94.0 req/s)
32 requests, single batch  :    26.2 ms  (1222.2 req/s)
Throughput speed-up        :  13.0x


## 5. 🔧 From scratch: a dynamic batching scheduler

The real production pattern: requests arrive at random times. A scheduler buffers
them and flushes the batch when **either** the batch reaches `max_batch_size`
**or** the oldest request has waited `max_delay_ms`. This bounds tail latency while
still amortising `c₀`. We'll simulate arrivals deterministically so the result is
reproducible.

In [5]:
@dataclass
class PendingRequest:
    arrival: float
    features: np.ndarray
    completed_at: float | None = None

class DynamicBatcher:
    def __init__(self, predict_fn, max_batch_size=16, max_delay=0.005):
        self.predict_fn = predict_fn
        self.max_batch_size = max_batch_size
        self.max_delay = max_delay

    def run(self, arrivals: list[PendingRequest], clock_start: float):
        """Process a list of timestamped arrivals using batch-or-timeout flushing.

        Returns per-request latencies. This is a discrete-event simulation of what
        an async scheduler does with a real event loop.
        """
        queue = sorted(arrivals, key=lambda r: r.arrival)
        latencies, i, now = [], 0, clock_start
        while i < len(queue):
            # Start a batch with the next available request
            batch_start_time = max(now, queue[i].arrival)
            batch = [queue[i]]; i += 1
            # Greedily add requests that have already arrived, up to limits
            while (i < len(queue)
                   and len(batch) < self.max_batch_size
                   and queue[i].arrival <= batch_start_time + self.max_delay):
                batch.append(queue[i]); i += 1
            # Run the batch
            X = np.array([r.features for r in batch])
            t0 = time.perf_counter(); self.predict_fn(X); compute = time.perf_counter() - t0
            finish = batch_start_time + compute
            for r in batch:
                latencies.append(finish - r.arrival)
            now = finish
        return np.array(latencies)

# Simulate 100 requests arriving over 0.5s (Poisson-ish, but seeded)
n = 100
inter = np.random.RandomState(1).exponential(0.005, n)
times = np.cumsum(inter)
arrivals = [PendingRequest(arrival=t, features=np.random.rand(2)) for t in times]

batcher = DynamicBatcher(timed_predict, max_batch_size=16, max_delay=0.005)
lat = batcher.run(arrivals, clock_start=0.0)
print(f"Dynamic batching over {n} requests:")
print(f"  p50 latency: {np.percentile(lat,50)*1000:6.1f} ms")
print(f"  p95 latency: {np.percentile(lat,95)*1000:6.1f} ms")
print(f"  p99 latency: {np.percentile(lat,99)*1000:6.1f} ms")
print(f"  max latency: {lat.max()*1000:6.1f} ms")

Dynamic batching over 100 requests:
  p50 latency:   11.3 ms
  p95 latency:   18.0 ms
  p99 latency:   18.7 ms
  max latency:   19.5 ms


## 6. 🧠 Tuning the trade-off: the latency/throughput frontier

There's no single 'best' setting — it depends on your SLO. Let's sweep
`max_delay` and watch p99 latency trade against achievable throughput. This curve
is the single most important picture in serving.

In [6]:
print(f"{'max_delay(ms)':>13} {'max_batch':>10} {'p99(ms)':>9} {'throughput(req/s)':>18}")
for delay in [0.001, 0.005, 0.010, 0.020]:
    for mbs in [8, 32]:
        b = DynamicBatcher(timed_predict, max_batch_size=mbs, max_delay=delay)
        lat = b.run(arrivals, clock_start=0.0)
        total_time = (arrivals[-1].arrival) + lat[-1]  # rough wall-clock span
        thru = n / total_time
        print(f"{delay*1000:>13.0f} {mbs:>10} {np.percentile(lat,99)*1000:>9.1f} {thru:>18.1f}")

max_delay(ms)  max_batch   p99(ms)  throughput(req/s)


            1          8      22.6              205.2


            1         32      22.6              205.2


            5          8      18.6              205.2


            5         32      18.5              205.2


           10          8      15.2              209.6


           10         32      15.2              209.5


           20          8      14.1              209.6


           20         32      15.1              209.6


Read the table as a *menu*, not a leaderboard. A bigger `max_delay`/`max_batch`
raises throughput (more amortisation of `c₀`) but pushes p99 latency up. If your
SLO is *"p99 < 50ms"*, you pick the highest-throughput row that still satisfies it.
This is the core capacity-planning decision for any online model service.

## 7. 🧠 The metrics that actually govern serving

You cannot operate what you don't measure. Online serving lives and dies by:

- **Throughput (QPS/RPS)** — requests served per second; sets your machine count.
- **Latency percentiles** — *never* report the mean. Report **p50/p95/p99**. The
  mean hides the tail, and the tail is what users feel. A 10ms mean with a 2s p99
  means 1% of users have a terrible time.
- **Error rate** — fraction of 5xx/4xx; the numerator of your availability SLO.
- **Saturation** — how full the queue/GPU is; the leading indicator of trouble.

These four (Google's "golden signals", minus a couple) are exactly the `/metrics`
counters we wired into `InferenceApp`, made quantitative.

In [7]:
def latency_report(lat_ms: np.ndarray):
    pcts = [50, 90, 95, 99, 99.9]
    print("Latency distribution:")
    for p in pcts:
        bar = "█" * int(np.percentile(lat_ms, p) / 2)
        print(f"  p{p:<5}: {np.percentile(lat_ms, p):6.1f} ms  {bar}")
    print(f"  mean : {lat_ms.mean():6.1f} ms  <-- note how the mean hides the tail")

latency_report(lat * 1000)

Latency distribution:
  p50   :    6.2 ms  ███
  p90   :   12.7 ms  ██████
  p95   :   13.2 ms  ██████
  p99   :   15.1 ms  ███████
  p99.9 :   15.2 ms  ███████
  mean :    5.5 ms  <-- note how the mean hides the tail


## 8. 🏭 In practice

| You built | Real-world equivalent |
|-----------|----------------------|
| `parse_http_request` / `build_http_response` | the HTTP layer inside FastAPI/uvicorn, gRPC |
| `InferenceApp` router + `/health` + `/metrics` | FastAPI app; KServe/Seldon predictor contract |
| `DynamicBatcher` | Triton Inference Server dynamic batching, TorchServe batching, vLLM continuous batching |
| latency percentiles + golden signals | Prometheus + Grafana dashboards, SLO alerting |

In a real stack you'd write the app with FastAPI and run it under uvicorn workers,
put it behind a load balancer, and let Triton or vLLM own the batching. But the
batch-or-timeout scheduler, the signature guard at the edge, and the p99-driven
capacity decision are all things you now understand from the inside.

```python
# The same /predict endpoint in FastAPI (shown for mapping, not run here):
# from fastapi import FastAPI
# app = FastAPI()
# @app.post("/predict")
# def predict(payload: dict):
#     X = np.atleast_2d(payload["instances"])
#     return {"predictions": (X @ W + B).tolist()}
# # run: uvicorn serve:app --workers 4 --port 8080
```

---

## 🎯 Exercises

Try these before moving on. They are deliberately open-ended — the goal is to
extend the machinery you just built, not to find a single right answer.

1. Add a `max_queue_size` to `DynamicBatcher`; when exceeded, reject new requests with a 503 (load shedding). Plot p99 latency with and without shedding under overload. Why is shedding kinder than unbounded queueing?
2. Our cost model is `c0 + b*c1`. Re-run the throughput sweep with `c0` set to near-zero (a cheap CPU model). Does batching still help? Explain when batching is *not* worth the latency cost.
3. Implement an A/B split in `InferenceApp`: route 10% of /predict traffic to a second model and tag the response with which model served it. This is the seed of canary deploys (Week 4).
4. The metrics endpoint reports cumulative counts. Add a rolling 1-minute QPS and error-rate window. Why are rolling windows, not cumulative totals, what alerting needs?
5. Add request-level logging that records (input hash, prediction, latency, model version). In Week 5 this log becomes your drift-detection and audit data source. What fields would you regret *not* logging?

---

## ✅ Recap — Week 3

- Serving exposes predict() over a network under a latency/availability contract; online is the hard case.
- An HTTP server is just byte parsing + routing + handlers — we built it before reaching for a framework.
- Fixed per-call overhead c0 is why batching multiplies throughput; dynamic batching trades a little latency for it.
- Tune batching against an SLO using the latency/throughput frontier, not a single 'best' number.
- Operate on p50/p95/p99 and the golden signals — the mean lies about the tail.

### Coming up next

**Week 4 — CI/CD & Testing for ML.** We've got a service. Now we automate the path *to* production: we'll build a pipeline DAG with caching from scratch, write the test types unique to ML (data validation, behavioral tests, the champion/challenger gate), then map it onto GitHub Actions.